# Semana 08 — Banco de Dados Relacionais e Não Relacionais

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 5, Semana 08 (abre o bloco final do módulo)

Esta semana muda de ambiente: até a Semana 07, você trabalhava só com arquivo (CSV, JSON, Excel) e, no máximo, um banco simples de arquivo único (`sqlite3`). A partir de agora, o Restaurante Sabor Caseiro passa a guardar os dados de verdade num banco de dados: você vai instalar o PostgreSQL na sua máquina, criar sua primeira tabela de verdade, e aprender a consultá-la com SQL — SELECT, filtros e agregações.

> Em cada tópico abaixo: **2 exemplos resolvidos** + **1 atividade prática** para você fazer sozinho(a).

### 🧭 De onde você vem: fechando o Bloco 4

Antes de entrar na Semana 08, vale reforçar de onde você está vindo e pra onde está indo dentro do curso.

**O que você viu e estudou na Semana 07:** o mesmo `dre_consolidado.csv` do Restaurante Sabor Caseiro, já limpo desde a Semana 06, virando gráfico com **Matplotlib** (linha, barra, dispersão, customização, subplots e exportação). Você também aprendeu a organizar código em **funções reutilizáveis** e conheceu sua primeira **classe** (`class`, `__init__`, `self`, método), e fechou com um **pipeline ETL** completo — Extract, Transform, Load — carregando o resultado num banco `sqlite3` e consultando com um primeiro `SELECT`.

**O que você fez:** criou gráficos customizados e exportados em `.png`, escreveu uma classe do zero, e montou 2 pipelines ETL de ponta a ponta (um salvando em CSV, outro num banco `sqlite3`), fechando o Mini Projeto do módulo — a Situação de Aprendizagem Integradora que uniu tudo dos Blocos 1 a 4.

**Qual era o objetivo:** te dar as ferramentas pra comunicar um dado já limpo (gráfico) e organizar o código que faz esse trabalho (funções, classes, pipeline) de ponta a ponta.

**Onde você está agora:** encerrando o Módulo Básico e abrindo o **Bloco 5** desta UC — o bloco final, sobre **Bancos de Dados, SQL e integração Python-PostgreSQL**. A Semana 08, que começa agora, resolve a pergunta que ficou em aberto no `sqlite3` da Semana 07: como guardar o dado num banco de verdade, instalar as ferramentas certas, e consultar esse dado com SQL de verdade — SELECT, filtros e agregações.

---
### 🔄 Retomada — o que vimos na Semana 07

Você já sabe transformar dado limpo em gráfico com Matplotlib, organizar código em funções e classes, e montar um pipeline ETL simples salvando o resultado num banco `sqlite3`. Esta semana usa essa base: o `SELECT * FROM despesas WHERE valor > 30000` que você rodou no fechamento da Semana 07 foi sua primeira consulta SQL — esta semana explica, do zero, como o banco guarda essa informação e como escrever esse tipo de consulta com confiança.

---
### 🟢 Abertura — Semana 08: Banco de Dados Relacionais e Não Relacionais

Começa o Bloco 5, o bloco final do curso! Nesta semana você entende <strong>como um banco de dados organiza a informação para análise (BI)</strong>, instala o PostgreSQL e o pgAdmin na sua máquina, cria sua primeira tabela de verdade, e escreve suas primeiras consultas SQL — do SELECT mais simples até agregações com GROUP BY.

**O que você vai aprender nesta semana:**
- Os fundamentos de Business Intelligence e os dois grandes tipos de modelagem de dados (relacional x dimensional), e OLTP x OLAP
- Instalar e configurar o PostgreSQL e o pgAdmin, e testar a conexão a partir do Python
- Escrever consultas básicas em SQL: SELECT, FROM, apelidos de coluna (AS), DISTINCT e LIMIT
- Filtrar resultados com WHERE, operadores de comparação, AND/OR/NOT, LIKE, IN, BETWEEN e IS NULL
- Resumir dados com funções de agregação (COUNT, SUM, AVG, MAX, MIN) e GROUP BY

### 📌 Antes de começar: ambiente e dataset desta semana

**Esta semana roda no VS Code local, não no Google Colab.** O Google Colab não mantém um serviço de banco de dados instalado e rodando entre sessões — e a partir de agora você vai conectar o Python a um PostgreSQL de verdade, instalado na sua própria máquina. Se você ainda não tem o VS Code e o Python configurados localmente, retome o `GUIA_GITHUB.md` (raiz do repositório) e o material da Semana 03 antes de continuar.

Esta semana usa um arquivo já conhecido do Restaurante Sabor Caseiro: `pedidos_sabor_caseiro.csv`, com 18 pedidos das 2 filiais já conhecidas (Centro e Zona Sul), com as colunas `id_pedido`, `cliente`, `filial`, `item`, `valor` e `data_hora`. Na Seção 2, você usa esse mesmo arquivo para criar e popular sua primeira tabela de verdade dentro do PostgreSQL — e é nela que você pratica todo SQL desta semana.

- `dataset/pedidos_sabor_caseiro.csv` — 18 linhas, um pedido por linha (a mesma tabela `pedidos` que você cria no banco na Seção 2)

---
## 1. Fundamentos de Business Intelligence e Modelagem de Dados

### O que é, afinal, um banco de dados — e o que é BI?

Até agora, todo dado que você usou no curso morava num **arquivo** — um CSV, um JSON, uma planilha Excel — que você abria com `pd.read_csv()` (ou equivalente), lia, e fechava. Um **banco de dados** é um programa diferente: fica **sempre ligado**, rodando em segundo plano, guardando o dado de forma organizada e permitindo que várias pessoas (ou vários programas, ao mesmo tempo) leiam e alterem essa informação sem esbarrar uma na outra.

**Business Intelligence (BI)** é o processo de transformar dado bruto em informação que ajuda alguém a decidir algo. Ele segue etapas parecidas com o que você já faz desde a Semana 06:

| Etapa do BI | O que acontece | O que você já fez, equivalente |
|---|---|---|
| Fonte de Dados | De onde o dado vem (banco, CSV, planilha) | `pd.read_csv()`, `pd.read_excel()` |
| ETL | Extrair, Transformar, Carregar | Limpeza de dados (Semana 06), pipeline ETL (Semana 07) |
| Modelagem/Análise | Organizar o dado pra ser consultado | Assunto desta seção e da Semana 09 |
| Visualização | Relatórios e dashboards | Gráficos com Matplotlib (Semana 07) |
| Compartilhamento | Entregar o conteúdo pra quem decide | O `.png` exportado, ou um dashboard publicado |

### Dois jeitos de modelar dado: relacional x dimensional

| | Relacional | Dimensional |
|---|---|---|
| Usado em | Bancos de Dados Relacionais (ex.: PostgreSQL) | Data Warehouse (DW) ou modelos de BI |
| Tipo de sistema | Transacional — OLTP | Analítico — OLAP |
| Nível de organização | Operacional (o dia a dia) | Tático/Estratégico (decisão) |
| Tipo de modelagem | Normalizado (menos repetição) | Desnormalizado (mais rápido de consultar) |
| Conceito-chave | Entidade / Relacionamento | Fato e Dimensão |
| Foco | Inserir, alterar e apagar dado com consistência | Consultar rápido um volume grande de dado |

O PostgreSQL que você instala nesta semana é um banco **relacional**, usado no dia a dia operacional do Sabor Caseiro (cada pedido sendo registrado). Isso não substitui o `dre_consolidado.csv` (o relatório mensal, já num formato mais próximo do dimensional) — os dois convivem, com propósitos diferentes.

### OLTP x OLAP

| Sigla | Nome completo | Para que serve | Exemplo no Sabor Caseiro |
|---|---|---|---|
| OLTP | *Online Transaction Processing* | Registrar operações do dia a dia, uma de cada vez, rápido | Cada pedido sendo lançado no caixa |
| OLAP | *Online Analytical Processing* | Analisar um grande volume de dados já acumulado, para tomar decisão | O relatório mensal consolidado que a diretoria usa |

### 🔹 Exemplo 1 — Representando um pedido como fato e dimensão

📖 **Antes do código:** numa modelagem dimensional, o evento que você quer medir (o pedido, com seu valor) vira uma linha de **fato**; quem/o quê/quando descreve esse evento vira **dimensão**. O código separa um pedido do CSV nesses dois papéis, usando dicionários Python.

In [ ]:
import pandas as pd

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")
linha = pedidos.iloc[0]

fato_pedido = {
    "id_pedido": int(linha["id_pedido"]),
    "valor": float(linha["valor"]),
}
dimensao_cliente = {"cliente": linha["cliente"]}
dimensao_item = {"item": linha["item"]}
dimensao_tempo = {"data_hora": linha["data_hora"]}

print("Fato (o que medimos):", fato_pedido)
print("Dimensão cliente:", dimensao_cliente)
print("Dimensão item:", dimensao_item)
print("Dimensão tempo:", dimensao_tempo)

### 🔹 Exemplo 2 — OLTP x OLAP na prática, com o mesmo arquivo

📖 **Antes do código:** uma consulta estilo OLTP busca **1 registro específico**, rápido — como o caixa conferindo um pedido. Uma consulta estilo OLAP resume **muitos registros de uma vez** — como a diretoria olhando o total vendido. `pedidos[pedidos["id_pedido"] == 3]` filtra 1 linha só (OLTP); `pedidos.groupby("filial")["valor"].sum()` soma o valor de TODOS os pedidos, agrupado por filial (OLAP) — o mesmo `groupby()` que você já usa desde a Semana 05.

In [ ]:
import pandas as pd

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

# Estilo OLTP: 1 registro específico, rápido
consulta_oltp = pedidos[pedidos["id_pedido"] == 3]
print("OLTP — pedido específico:")
display(consulta_oltp)

# Estilo OLAP: resumo sobre muitos registros
consulta_olap = pedidos.groupby("filial")["valor"].sum().reset_index()
print("\nOLAP — total vendido por filial:")
display(consulta_olap)

### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** o dono do Sabor Caseiro quer entender, em números simples, a diferença entre "olhar 1 pedido" e "olhar o negócio inteiro" antes de começar a mexer no banco de dados de verdade.

**Comando:** usando `pedidos_sabor_caseiro.csv`, faça (1) uma consulta estilo OLTP que busque só os pedidos do cliente `"Marcos Silva"`, e (2) uma consulta estilo OLAP que calcule a média (`.mean()`) do valor dos pedidos, agrupada por `item`.

In [ ]:
import pandas as pd

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

# OLTP: pedidos de um cliente específico
pedidos_marcos = pedidos[pedidos["cliente"] == "Marcos Silva"]
print("OLTP — pedidos do Marcos Silva:")
display(pedidos_marcos)

# OLAP: valor médio por item
media_por_item = pedidos.groupby("item")["valor"].mean().reset_index()
print("\nOLAP — valor médio por item:")
display(media_por_item)

---
## 2. PostgreSQL e pgAdmin — Instalação, Configuração e Primeira Conexão

Até aqui você só instalou **bibliotecas** Python (`pandas`, `openpyxl`...) — bastava um `%pip install` e pronto, dentro do próprio notebook. O PostgreSQL é diferente: é um **programa completo**, separado do Python, que fica rodando sozinho em segundo plano na sua máquina (chamado de **servidor**) esperando alguém — o pgAdmin, o Python, ou qualquer outro programa — se conectar nele.

- **PostgreSQL** — o SGBD (Sistema Gerenciador de Banco de Dados): o programa que guarda e organiza os dados de verdade. Um único PostgreSQL pode hospedar **vários bancos de dados diferentes** — ele já vem com um banco chamado `postgres`, e você cria um outro, separado, chamado `sabor_caseiro`.
- **pgAdmin** — um programa à parte, com tela e botões, que te deixa ver e mexer no PostgreSQL sem digitar comando nenhum.
- **psycopg2** — a biblioteca Python que serve de ponte entre o seu código e o PostgreSQL.
- **SQL** — a linguagem usada para pedir e alterar dado dentro de um banco relacional. Você já viu uma prévia dela no fechamento da Semana 07; esta semana você aprofunda a sintaxe de verdade.

### Passo a passo — instalação do PostgreSQL (fora do notebook, Windows)

1. **Baixe o instalador oficial** em `https://www.postgresql.org/download/windows/` — clique em "Download the installer". Escolha a versão mais recente disponível.
2. **Rode o instalador** e siga o assistente. Mantenha marcados os componentes padrão: PostgreSQL Server, pgAdmin 4 e Command Line Tools.
3. O instalador pede uma **senha para o usuário `postgres`** — o usuário administrador padrão. Anote essa senha: você vai digitá-la toda vez que conectar.
4. Mantenha a **porta padrão, `5432`**.
5. Ao final, o instalador pode abrir o **Stack Builder** — você pode fechar essa janela sem problema.
6. O PostgreSQL fica rodando automaticamente como um **serviço do Windows**. Para confirmar: abra o menu Iniciar, digite **Serviços** (ou `services.msc`), procure `postgresql-x64-...` e confira se o Status diz **"Em execução"**.

> ⚠️ **Confusão comum:** o PostgreSQL vem com um usuário chamado `postgres` **e**, separadamente, com um banco de dados que também se chama `postgres` — são 2 coisas diferentes com o mesmo nome. Você não usa esse banco padrão para nada: cria um banco novo, `sabor_caseiro`.

### Primeiro acesso ao pgAdmin e criação do banco `sabor_caseiro`

1. Abra o **pgAdmin 4**.
2. Na primeira vez, ele pede uma **Master Password** — uma senha nova, exclusiva do pgAdmin. É diferente da senha do usuário `postgres`.
3. No painel esquerdo, expanda **Servers** → clique no servidor pré-cadastrado → digite a senha do usuário `postgres` quando pedida.
4. Clique com o **botão direito em "Databases"** → **Create** → **Database...** → em "Database", digite `sabor_caseiro` → **Save**.
5. Confira: o banco `sabor_caseiro` precisa aparecer na árvore à esquerda.

> ⚠️ Se a instalação travar ou você não conseguir instalar a tempo, avise o professor — existe uma alternativa de PostgreSQL gratuito na nuvem (Supabase, Neon) para continuar acompanhando a aula.

### 🔹 Exemplo 1 — Testando a conexão com Python

📖 **Antes do código:** `psycopg2.connect(...)` abre uma conexão com o banco, recebendo `host`, `dbname`, `user` e `password`. `conexao.cursor()` cria um **cursor** — o objeto usado para executar comandos SQL. `cursor.execute("SELECT version();")` roda esse comando, e `cursor.fetchone()` busca a primeira linha do resultado.

In [ ]:
%pip install psycopg2-binary

try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("SELECT version();")
    print("Conectado! Versão do PostgreSQL:", cursor.fetchone()[0])
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install acima e execute de novo.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

> ⚠️ **Veja como é um erro real do Python.** O `except psycopg2.OperationalError` acima existe porque, sem o PostgreSQL rodando (ou com a senha errada), é exatamente esse erro que aparece — algo como `connection to server at "localhost" (...), port 5432 failed: Connection refused`. Rodar essa célula sem o PostgreSQL instalado ainda é normal: é assim que você confirma, mais tarde, que o setup funcionou.

### 🔹 Exemplo 2 — Criando e populando sua primeira tabela de verdade

📖 **Antes do código:** `CREATE TABLE` cria uma tabela nova, definindo o nome de cada coluna e seu tipo (você estuda isso a fundo na Semana 09 — por enquanto, é só o suficiente pra ter uma tabela real pra consultar esta semana). `SERIAL PRIMARY KEY` cria uma coluna numérica que se autoincrementa, servindo de identificador único. `INSERT INTO ... VALUES (...)` insere uma linha por vez. `%s` é um marcador de posição que o psycopg2 preenche com segurança (evita o erro de digitar aspas erradas dentro do SQL), e `cursor.executemany()` roda o mesmo INSERT várias vezes, uma por linha da lista `dados`. `conexao.commit()` confirma a gravação — sem ele, as linhas inseridas não ficam salvas de verdade no banco.

In [ ]:
import pandas as pd

try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pedidos (
            id_pedido SERIAL PRIMARY KEY,
            cliente VARCHAR(100),
            filial VARCHAR(50),
            item VARCHAR(100),
            valor NUMERIC(10,2),
            data_hora TIMESTAMP
        );
    """)

    cursor.execute("TRUNCATE TABLE pedidos RESTART IDENTITY;")

    pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")
    dados = list(pedidos[["cliente", "filial", "item", "valor", "data_hora"]].itertuples(index=False, name=None))
    cursor.executemany(
        "INSERT INTO pedidos (cliente, filial, item, valor, data_hora) VALUES (%s, %s, %s, %s, %s);",
        dados,
    )
    conexao.commit()

    cursor.execute("SELECT COUNT(*) FROM pedidos;")
    print("Tabela pedidos criada e populada! Total de linhas:", cursor.fetchone()[0])
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install do Exemplo 1 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** antes de seguir para as consultas SQL, o dono do restaurante quer ter certeza de que a tabela `pedidos` está de pé e com o número certo de linhas — sem isso, nenhuma das próximas seções funciona.

**Comando:** instale o PostgreSQL e o pgAdmin, crie o banco `sabor_caseiro`, rode o Exemplo 2 (trocando `"SUA_SENHA_AQUI"` pela sua senha) até ver "Tabela pedidos criada e populada! Total de linhas: 18", e depois rode a célula abaixo pra listar as tabelas existentes no banco (repita o padrão do Exemplo 1, com uma consulta diferente).

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';")
    tabelas = cursor.fetchall()
    print("Tabelas encontradas:", tabelas)
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install do Exemplo 1 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

### ✅ Antes de seguir para a Seção 3, confirme:

- [ ] Nos Serviços do Windows (`services.msc`), `postgresql-x64-...` aparece como **"Em execução"**
- [ ] O banco `sabor_caseiro` aparece na árvore do pgAdmin, dentro de "Databases"
- [ ] A tabela `pedidos` existe e tem 18 linhas (Exemplo 2 imprimiu isso)
- [ ] A Atividade Prática 2 listou a tabela `pedidos` em `information_schema.tables`

Se algum item falhou, volte ao passo a passo antes de continuar — as Seções 3, 4 e 5 dependem de a tabela `pedidos` já existir.

---
## 3. Queries Básicas no PostgreSQL

Toda consulta SQL de leitura começa com o mesmo par de palavras-chave: **SELECT** (o que eu quero ver) e **FROM** (de onde vem). A partir daí, alguns recursos deixam a consulta mais precisa e mais legível:

| Comando | O que faz |
|---|---|
| `SELECT * FROM tabela` | Traz TODAS as colunas |
| `SELECT coluna1, coluna2 FROM tabela` | Traz só as colunas escolhidas |
| `SELECT coluna AS apelido FROM tabela` | Renomeia a coluna no resultado (sem mudar o banco) |
| `SELECT DISTINCT coluna FROM tabela` | Remove valores repetidos do resultado |
| `... LIMIT N` | Traz só as N primeiras linhas |
| `... ORDER BY coluna` | Ordena o resultado (ASC crescente, padrão; DESC decrescente) |

### 🔹 Exemplo 1 — SELECT com colunas específicas e apelido (AS)

📖 **Antes do código:** a consulta abaixo escolhe só 3 colunas da tabela `pedidos`, e usa `AS` para renomear `cliente` como `nome_cliente` no resultado — o nome da coluna no banco continua `cliente`, só o resultado da consulta é que aparece com outro rótulo. `LIMIT 5` garante que só as 5 primeiras linhas voltam.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT id_pedido, cliente AS nome_cliente, valor
        FROM pedidos
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — DISTINCT e ORDER BY

📖 **Antes do código:** `SELECT DISTINCT filial FROM pedidos` traz cada valor de `filial` **uma única vez**, mesmo que existam 18 pedidos espalhados entre só 2 filiais. `ORDER BY filial` ordena o resultado alfabeticamente.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT DISTINCT filial
        FROM pedidos
        ORDER BY filial;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 3 — Sua vez de programar

**Contextualização:** a equipe de atendimento quer uma lista enxuta dos pedidos mais recentes, só com as colunas que interessam pra eles, com nomes fáceis de entender.

**Comando:** escreva uma consulta SQL que traga as colunas `item`, `valor AS preco` e `data_hora`, ordenada por `data_hora` em ordem decrescente (`DESC`), trazendo só as 5 primeiras linhas.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT item, valor AS preco, data_hora
        FROM pedidos
        ORDER BY data_hora DESC
        LIMIT 5;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


---
## 4. Filtros no PostgreSQL

O comando **WHERE** filtra as linhas de uma consulta, mantendo só as que atendem a uma condição.

| Operador/Comando | O que faz | Exemplo |
|---|---|---|
| `=`, `<>`, `>`, `<`, `>=`, `<=` | Comparação | `WHERE valor > 30` |
| `AND` / `OR` / `NOT` | Combina condições | `WHERE filial = 'Centro' AND valor > 30` |
| `LIKE '%texto%'` | Busca um texto dentro de outro (case-sensitive no PostgreSQL) | `WHERE item LIKE '%Suco%'` |
| `IN (...)` | O valor está numa lista? | `WHERE item IN ('Suco Natural', 'Sobremesa do Dia')` |
| `BETWEEN a AND b` | O valor está num intervalo? | `WHERE valor BETWEEN 10 AND 40` |
| `IS NULL` / `IS NOT NULL` | O valor está vazio (ou não)? | `WHERE cliente IS NOT NULL` |

> ⚠️ **Cuidado: o PostgreSQL é *case-sensitive*.** `WHERE filial = 'centro'` (minúsculo) NÃO encontra as linhas onde `filial` está gravado como `'Centro'` (maiúsculo). O texto do filtro precisa bater exatamente com o que está gravado.

### 🔹 Exemplo 1 — WHERE com comparação e AND

📖 **Antes do código:** a consulta traz só os pedidos da filial Centro com valor maior que 30 — as duas condições, unidas por `AND`, precisam ser verdadeiras ao mesmo tempo para a linha aparecer no resultado.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, filial, item, valor
        FROM pedidos
        WHERE filial = 'Centro' AND valor > 30
        ORDER BY valor DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — LIKE, IN e BETWEEN

📖 **Antes do código:** `item LIKE '%Suco%'` encontra qualquer linha em que `item` contenha a palavra "Suco" em qualquer posição — os `%` são curingas que representam "qualquer texto antes/depois". `valor BETWEEN 10 AND 40` mantém só os valores dentro desse intervalo, incluindo as pontas.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, item, valor
        FROM pedidos
        WHERE item LIKE '%Suco%' OR valor BETWEEN 10 AND 40
        ORDER BY valor;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 4 — Sua vez de programar

**Contextualização:** a gerência da filial Zona Sul quer saber quais pedidos dessa filial passaram de R$ 40 — um relatório rápido de "pedidos grandes" para conferir se algum precisa de atenção especial (item em falta, cliente recorrente etc.).

**Comando:** escreva uma consulta SQL com WHERE que traga `cliente`, `item` e `valor` só dos pedidos da filial `'Zona Sul'` com `valor` maior que 40, ordenados do maior valor para o menor.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT cliente, item, valor
        FROM pedidos
        WHERE filial = 'Zona Sul' AND valor > 40
        ORDER BY valor DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


---
## 5. Funções de Agregação e Agrupamentos

Funções de agregação calculam **um valor a partir de várias linhas**.

| Função | O que faz |
|---|---|
| `COUNT(*)` | Conta quantas linhas existem |
| `SUM(coluna)` | Soma os valores da coluna |
| `AVG(coluna)` | Calcula a média |
| `MAX(coluna)` / `MIN(coluna)` | O maior e o menor valor |

Sozinha, uma função de agregação resume a tabela INTEIRA numa única linha. Para resumir **por grupo** (por filial, por cliente...), existe o **GROUP BY**: ele separa as linhas em grupos, e a função de agregação passa a calcular dentro de cada grupo, não mais da tabela toda.

### 🔹 Exemplo 1 — Agregação simples, sem GROUP BY

📖 **Antes do código:** `COUNT(*)` conta todas as linhas da tabela; `SUM(valor)` soma a coluna `valor` inteira; `AVG(valor)` calcula a média. Sem `GROUP BY`, o resultado é sempre 1 única linha, resumindo a tabela toda.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT COUNT(*) AS total_pedidos,
               SUM(valor) AS faturamento_total,
               AVG(valor) AS ticket_medio
        FROM pedidos;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### 🔹 Exemplo 2 — GROUP BY: agregação por grupo

📖 **Antes do código:** `GROUP BY filial` separa as linhas em 2 grupos (Centro e Zona Sul) antes de calcular `COUNT(*)` e `SUM(valor)` — o resultado tem 1 linha por filial, não 1 linha por pedido. `ORDER BY faturamento DESC` ordena os grupos do maior faturamento para o menor.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT filial,
               COUNT(*) AS qtd_pedidos,
               SUM(valor) AS faturamento
        FROM pedidos
        GROUP BY filial
        ORDER BY faturamento DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


### ✏️ Atividade Prática 5 — Sua vez de programar

**Contextualização:** o dono do restaurante quer saber qual item é o mais vendido em quantidade, e qual o valor médio de cada item — informação que ele vai usar pra decidir o que destacar no cardápio do mês.

**Comando:** escreva uma consulta SQL que agrupe por `item`, trazendo `COUNT(*) AS qtd_vendida` e `AVG(valor) AS valor_medio`, ordenada por `qtd_vendida` em ordem decrescente.

In [ ]:
import psycopg2

try:
    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        SELECT item,
               COUNT(*) AS qtd_vendida,
               AVG(valor) AS valor_medio
        FROM pedidos
        GROUP BY item
        ORDER BY qtd_vendida DESC;
    """)
    colunas = [desc[0] for desc in cursor.description]
    resultado = cursor.fetchall()
    import pandas as pd
    display(pd.DataFrame(resultado, columns=colunas))
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda nao foi instalado nesta sessao — rode a celula %pip install da Secao 2 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda nao conectou. Confira, nos Servicos do Windows, se 'postgresql-x64-...' esta 'Em execucao', e se host/usuario/senha/banco estao corretos.")
    print(f"Erro real do Python: {erro}")


---
## 6. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe um cenário de negócio diferente, com uma tabela já pronta (criada e populada pra você) — o desafio é escrever a consulta SQL que responde à pergunta de negócio, usando SELECT, WHERE, ORDER BY, e funções de agregação com GROUP BY quando fizer sentido.

### Squad B — Loja de Roupas Online

**Contextualização:** a loja quer saber quais pedidos passaram de R$ 100, pra entender o perfil de compra dos clientes que gastam mais.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_b_pedidos (
            cliente VARCHAR(100), produto VARCHAR(100), valor NUMERIC(10,2)
        );
    """)
    cursor.execute("TRUNCATE TABLE squad_b_pedidos;")
    cursor.executemany(
        "INSERT INTO squad_b_pedidos (cliente, produto, valor) VALUES (%s, %s, %s);",
        [
            ("Beatriz Lima", "Camiseta P", 59.90),
            ("Beatriz Lima", "Calça Jeans M", 129.90),
            ("Diego Rocha", "Jaqueta G", 219.90),
            ("Diego Rocha", "Boné", 39.90),
        ],
    )
    conexao.commit()
    print("Tabela squad_b_pedidos pronta!")

    # escreva aqui a consulta que traz os pedidos com valor > 100, ordenados do maior para o menor
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Squad C — Plataforma de Streaming de Música

**Contextualização:** a plataforma quer saber quantas vezes cada playlist aparece na tabela de reprodução, pra entender quais playlists são mais montadas pelos usuários.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_c_playlists (
            playlist VARCHAR(100), musica VARCHAR(100)
        );
    """)
    cursor.execute("TRUNCATE TABLE squad_c_playlists;")
    cursor.executemany(
        "INSERT INTO squad_c_playlists (playlist, musica) VALUES (%s, %s);",
        [
            ("Foco no Trabalho", "Lo-fi Beats 01"),
            ("Foco no Trabalho", "Lo-fi Beats 02"),
            ("Relaxar à Noite", "Lo-fi Beats 01"),
            ("Treino", "Corrida Intensa"),
        ],
    )
    conexao.commit()
    print("Tabela squad_c_playlists pronta!")

    # escreva aqui a consulta que conta quantas músicas cada playlist tem (GROUP BY playlist)
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Squad D — Clínica Médica

**Contextualização:** a clínica quer buscar o prontuário de um paciente específico pelo nome, pra conferir o tipo sanguíneo antes de um procedimento.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_d_pacientes (
            paciente VARCHAR(100), numero_prontuario VARCHAR(20), tipo_sanguineo VARCHAR(5)
        );
    """)
    cursor.execute("TRUNCATE TABLE squad_d_pacientes;")
    cursor.executemany(
        "INSERT INTO squad_d_pacientes (paciente, numero_prontuario, tipo_sanguineo) VALUES (%s, %s, %s);",
        [
            ("Otávio Ramos", "PRT-1042", "O+"),
            ("Vanessa Melo", "PRT-1043", "A-"),
        ],
    )
    conexao.commit()
    print("Tabela squad_d_pacientes pronta!")

    # escreva aqui a consulta que busca o paciente "Vanessa Melo" (WHERE paciente = ...)
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### Squad E — Aplicativo de Delivery

**Contextualização:** o aplicativo quer saber o total gasto por cada usuário, pra identificar os clientes mais fiéis do mês.

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost", dbname="sabor_caseiro", user="postgres", password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS squad_e_pedidos (
            usuario VARCHAR(100), restaurante VARCHAR(100), valor NUMERIC(10,2)
        );
    """)
    cursor.execute("TRUNCATE TABLE squad_e_pedidos;")
    cursor.executemany(
        "INSERT INTO squad_e_pedidos (usuario, restaurante, valor) VALUES (%s, %s, %s);",
        [
            ("Renata Duarte", "Pizzaria Napoli", 68.00),
            ("Renata Duarte", "Sabor Caseiro - Centro", 32.90),
            ("Thiago Nogueira", "Sabor Caseiro - Centro", 45.00),
            ("Thiago Nogueira", "Pizzaria Napoli", 52.00),
        ],
    )
    conexao.commit()
    print("Tabela squad_e_pedidos pronta!")

    # escreva aqui a consulta que soma o valor gasto por usuário (GROUP BY usuario), ordenada do maior pro menor
    cursor.execute("""
        -- SUA CONSULTA AQUI
    """)
    print(cursor.fetchall())
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão.")
except psycopg2.OperationalError as erro:
    print(f"Erro real do Python: {erro}")

### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Algum squad precisou combinar WHERE com GROUP BY na mesma consulta? Nessa ordem, ou trocada?
- Algum squad usou LIKE ou IN sem perceber que precisava?
- O que aconteceria se um squad esquecesse o `ORDER BY` numa consulta pedida "do maior pro menor"?

---
### 🏁 Fechamento — Semana 08

**Nesta semana você aprendeu:**
- Os fundamentos de BI, modelagem relacional x dimensional, e OLTP x OLAP
- Instalar e configurar o PostgreSQL e o pgAdmin, e criar sua primeira tabela de verdade
- Escrever consultas com SELECT, FROM, AS, DISTINCT, LIMIT e ORDER BY
- Filtrar resultados com WHERE, AND/OR, LIKE, IN, BETWEEN e IS NULL
- Resumir dados com COUNT, SUM, AVG, MAX, MIN e GROUP BY

**Próxima semana:** a Semana 09 aprofunda em SQL, unindo tabelas diferentes com JOIN e salvando consultas prontas com VIEW.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 08 — Banco de Dados Relacionais e Não Relacionais

*Prof. Especialista Cláudio F. Neves*